In [ ]:
import json
import re

from langchain_community.document_loaders import JSONLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
loader = JSONLoader(
    file_path="/home/mlin/repos/scratch/finance-rag-assistant/scripts/data/normalized/AMZN_000101872425000004/text.jsonl",
    jq_schema=".text",
    text_content=True,
    json_lines=True,
)

docs = loader.load()
# print(docs[0])

In [ ]:
def clean_text(t: str) -> str:
    t = re.sub(r"\s+\n", "\n", t)
    t = re.sub(r"\n{3,}", "\n\n", t)
    return t.strip()
    
splitter = RecursiveCharacterTextSplitter(
    chunk_size=3200,
    chunk_overlap=400,  # characters ~ ~800–1000 tokens
    separators=["\n\n", "\n", " ", ""],
)
records = []
for d in docs:
    meta = d.metadata
    with open(meta["source"], "r") as f:
        file = json.load(f)
        meta |= {k: file[k] for k in ["doc_id", "ticker", "cik", "filing_date", "source_url"]}
    for chunk in splitter.split_text(clean_text(d.page_content)):
        records.append({"text": chunk, "metadata": meta})

In [ ]:
len(records)

# TO CHECK: why no overlap between adjacent records?

In [ ]:
records[20]

In [ ]:
records[21]

In [ ]:
# !cd ..
# !source .venv/bin/activate
# !poetry add jq
# !poetry install